In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../')
import glob
import matplotlib.lines as mlines
from matplotlib.colors import Normalize, LogNorm
import cmocean
from tqdm.notebook import tqdm
from datetime import datetime, timedelta

In [ ]:
path = f'/scratch/projects/phase01/rab/B06/PROCESSED_ERA5/'

months_array = np.arange(1,12+1)
months = [f"{m:02d}" for m in months_array]

filenames = sorted(glob.glob(path + 'Composites/CompositesInversion_2017*_0.zarr'))
composites = xr.open_mfdataset(filenames, engine = 'zarr', combine = 'nested', concat_dim = 'time').sel(longitude = slice(-30,30)).sortby('time')

In [ ]:
R_d = 287.05       # J kg-1 K-1
g = 9.80665        # m s-2
H = 7000

def temperature_from_gph(z_wave, height):

    z_wave = z_wave.transpose(...,"level", "longitude")
    height = height.transpose(...,"level", "longitude")

    levels = z_wave.level

    z_plus = z_wave.isel(level=slice(2, None))
    z_minus = z_wave.isel(level=slice(None, -2))

    h_plus = height.isel(level=slice(2, None))
    h_minus = height.isel(level=slice(None, -2))

    interior_levels = levels.isel(level=slice(1, -1))

    z_plus = z_plus.assign_coords(level=interior_levels)
    z_minus = z_minus.assign_coords(level=interior_levels)

    h_plus = h_plus.assign_coords(level=interior_levels)
    h_minus = h_minus.assign_coords(level=interior_levels)

    dZ = z_plus - z_minus
    dz = h_plus - h_minus

    T_interior = (g * H / R_d) * dZ / dz

    z0 = z_wave.isel(level=0)
    z1 = z_wave.isel(level=1)

    h0 = height.isel(level=0)
    h1 = height.isel(level=1)

    T_bottom = (
        (g * H / R_d)
        * (z1 - z0)
        / (h1 - h0)
    ).expand_dims(level=[levels.values[0]])

    zm1 = z_wave.isel(level=-2)
    ztop = z_wave.isel(level=-1)

    hm1 = height.isel(level=-2)
    htop = height.isel(level=-1)

    T_top = (
        (g * H / R_d)
        * (ztop - zm1)
        / (htop - hm1)
    ).expand_dims(level=[levels.values[-1]])

    T_wave = xr.concat(
        [T_bottom, T_interior, T_top],
        dim="level"
    )

    return T_wave

def o3_to_ppbv(o3):
    M_air = 28.9644   # g/mol
    M_o3 = 48.00      # g/mol

    return o3 * (M_air / M_o3) * 1e9

def specific_humidity_to_ppmv(q):
    M_h2o = 18.01528  # g/mol
    M_dry = 28.9644   # g/mol

    x_h2o = (q / M_h2o) / (
        q / M_h2o + (1 - q) / M_dry
    )

    return x_h2o * 1e6

In [ ]:
ratio_data = composites.sel(box_center = slice(70,130)).sel(lat_center = 0)
print(len(ratio_data.time))
valid_mask = ratio_data.valid_days > 0
print(valid_mask.sum().values)
drop_mask = ratio_data.center_method == 'tropopause_jump'
print(drop_mask.sum().values)

print(drop_mask.sum().values/valid_mask.sum().values*100)

In [ ]:
#seasonal_mean = composites.sel(box_center = 110).groupby("time.season").mean(dim="time").sel(lat_center = 0)
#composite_plot = seasonal_mean.sel(season="DJF")

valid_mask = composites.valid_days > 0
drop_mask = composites.center_method == 'Ri'#
composites_filtered = composites.where(drop_mask)
# mask = composites.valid_days > 0
# composites = composites.where(mask.compute())


composite_plot = composites_filtered.sel(box_center = slice(70,130)).mean(dim = 'box_center').sel(lat_center = 0).drop_dims('day')#.sel(box_center = slice(90,150)).mean(dim = 'box_center')
composite_plot = composite_plot.mean(dim = 'time')#.groupby('time.season').mean(dim = 'time').sel(season = 'DJF')
height_plot = composite_plot.height

S_0 = composite_plot.Ri
lrt_day = composite_plot.z_lrt
fig, axes = plt.subplots(
    5, 2,
    figsize=(12, 10),#
    sharey=True,
    constrained_layout=True
)

o3_ppbv = o3_to_ppbv(composite_plot.o3)

o3_gradient = (
    o3_ppbv.differentiate(coord="level")
    / composite_plot.height.differentiate(coord="level")
    * 1000
)


fields = [
    composite_plot.u - composite_plot.u_perturbation,
    composite_plot.u,
    composite_plot.u_perturbation,
    composite_plot.u_Kelvin,
    composite_plot.t_perturbation,
    temperature_from_gph(composite_plot.z_Kelvin, composite_plot.height),
    composite_plot.t,
    specific_humidity_to_ppmv(composite_plot.q),
    o3_ppbv,
    o3_gradient,
]

titles = [
    r"$\bar{u}$",
    r"$u$",
    r"$u'$",
    r"$u'_{Kelvin}$",
    r"$T'$",
    r"$T'_{Kelvin}$",
    r"T",
    r"$H_2O$",
    r"$O_3$",
    r"$\frac{\partial O_3}{\partial z}$",
]

cmaps = [
    "seismic",
    "seismic",
    "seismic",
    "seismic",
    cmocean.cm.balance,
    cmocean.cm.balance,
    "coolwarm",
    "YlGnBu",
    cmocean.cm.haline,
    "Spectral_r",
]

norms = [
    Normalize(-20, 20),
    Normalize(-20, 20),
    Normalize(-20, 20),
    Normalize(-20, 20),
    Normalize(-3, 3),
    Normalize(-3, 3),
    Normalize(190, 200),
    Normalize(3.5,5),
    Normalize(80, 300),
    Normalize(0, 300),
]


contours = [
    None,
    None,
    None,
    None,
    None,
    None,
    None,
    None,
    None,
    None,
]

contours_clouds = [
    True,
    None,
    None,
    None,
    None,
    None,
    None,
    None,
    None,
    None,
]
pcm_wind = None
pcm_temp = None

Lon= np.broadcast_to(composite_plot.longitude.values, height_plot.shape)


panel_labels = ["a)", "b)", "c)", "d)", "e)", "f)", "g)", "h)", "i)", "j)"]

pcm_list = []

for i, (ax, field, title, shear, cloud, cmap, norm) in enumerate(zip(
    axes.flat, fields, titles, contours, contours_clouds, cmaps, norms
    )):


    pcm = ax.pcolormesh(
        composite_plot.longitude,
        height_plot,
        field,
        cmap=cmap,
        norm=norm,
        shading="auto",
    )

    if shear is not None:
        ax.contour(
            Lon,
            height_plot,
            shear,
            levels=[1,3,6],
            colors = ['yellow', 'orange', 'red'],
            linewidths=3,
        )
    if cloud is not None:
        ax.contour(
                Lon,
                height_plot,
                composite_plot.cc,
                cmap = cmocean.cm.rain,
                levels = [0.1,0.2,0.3,0.4,0.5],
                linewidths=3,
        )
    shear_handle = mlines.Line2D(
        [], [], 
        color="yellow",
        linewidth=3,
        label="Ri < 1"
    )

    ax.plot(
        composite_plot.longitude,
        lrt_day,
        "k--",
        lw=2,
        label="LRT"
    )

    # ax.plot(
    #     composite_plot.longitude,
    #     composite_plot.cpt_height,
    #     "k--",
    #     lw=2,
    #     label="LRT"
    # )

    ax.text(
        0.02, 0.97,
        panel_labels[i],
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
        ha="left",
    )

    ax.set_title(title, fontsize=14)
    if i > 7:
        ax.set_xlabel("Relative Longitude (°)", fontsize = 12)
    else:
        ax.set_xticklabels([])
    if i in [0,2,4,6,8]:
        ax.set_ylabel("Height (m)", fontsize = 12)
    ax.grid(alpha=0.3)
    #ax.set_xlim(0,150)
    ax.set_ylim(10000,20000)
    pcm_list.append(pcm)

handles, labels = ax.get_legend_handles_labels()
handles.append(shear_handle)

#axes[0].set_ylabel("Height (m)")
#axes[0].legend(handles = handles)


cbar1 = fig.colorbar(
    pcm_list[0],
    ax=axes[0, :],
    orientation="vertical",
    shrink=0.9,
    pad=0.02,
)
cbar1.set_label(r"Zonal wind [m s$^{-1}$]", fontsize = 11)

cbar2 = fig.colorbar(
    pcm_list[2],
    ax=axes[1, :],
    orientation="vertical",
    shrink=0.9,
    pad=0.02,
)
cbar2.set_label(r"Zonal wind [m s$^{-1}$]", fontsize = 11)

cbar3 = fig.colorbar(
    pcm_list[4],
    ax=axes[2, :],
    orientation="vertical",
    shrink=0.9,
    pad=0.02,
)
cbar3.set_label("Temperature perturbation [K]", fontsize = 11)

# Individual colorbars for the final four panels
cbar_labels = [
    "Temperature [K]",
    "Water vapor [ppmv]",
    "Ozone [ppbv]",
    r"Ozone gradient [ppbv km$^{-1}$]",
]

for j, label in enumerate(cbar_labels):
    ax = axes.flat[j + 6]

    cbar = fig.colorbar(
        pcm_list[j + 6],
        ax=ax,
        orientation="vertical",
        fraction=0.046,
        pad=0.03,
    )
    cbar.set_label(label, fontsize=11)
    cbar.ax.tick_params(labelsize=9)

plt.savefig('../../Plots/PaperReady/AppendixComposite.png', format = 'PNG', dpi = 300)

In [ ]:
#print(composites_filtered.z_lrt.sel(box_center = slice(60,130)).sel(longitude = 0))
z_1 = composites_filtered.z_lrt.sel(box_center = slice(60,130)).sel(longitude = 0).compute().values.flatten()
z_2 = composites_filtered.z_lrt.sel(box_center = slice(60,130)).sel(longitude = -20).compute().values.flatten()
z_1 = z_1[np.isfinite(z_1)]
z_2 = z_2[np.isfinite(z_2)]

plt.figure(figsize=(9, 5))

bins = np.arange(14000,18501, 250)
plt.hist(z_1, bins=bins, alpha=0.6, label='Relative Longitude 0°', edgecolor='black')
plt.hist(z_2, bins=bins, alpha=0.6, label='Relative Longitude -20°', edgecolor='black')

plt.xlabel('Tropopause Height [m]')
plt.ylabel('Frequency')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.vlines(x = 16000, ymin = 0, ymax = 400, color = 'black', linestyle = '--')
plt.ylim(0,400)
plt.tight_layout()

#plt.savefig('../../Plots/PaperReady/AppendixTropCriterion.png', format = 'PNG', dpi = 300)

In [ ]:
print(len(z_1))
print((z_1 > 16000).sum())
print((z_1 > 16000).sum()/len(z_1))
print(len(z_2))
print((z_2 < 16000).sum())
print((z_2 < 16000).sum()/len(z_2))

In [ ]:
composite = composites.sel(box_center = 110).sel(lat_center = 0).compute()

valid_mask = composite.valid_days > 0

composite_plot = composite.where(valid_mask, drop = True).isel(time = 132)

height_plot = composite_plot.height

S_0 = composite_plot.Ri
lrt_day = composite_plot.z_lrt
fig, axes = plt.subplots(
    5, 2,
    figsize=(12, 10),#
    sharey=True,
    constrained_layout=True
)

o3_ppbv = o3_to_ppbv(composite_plot.o3)

o3_gradient = (
    o3_ppbv.differentiate(coord="level")
    / composite_plot.height.differentiate(coord="level")
    * 1000
)


fields = [
    composite_plot.u - composite_plot.u_perturbation,
    composite_plot.u,
    composite_plot.u_perturbation,
    composite_plot.u_Kelvin,
    composite_plot.t_perturbation,
    temperature_from_gph(composite_plot.z_Kelvin, composite_plot.height),
    composite_plot.t,
    specific_humidity_to_ppmv(composite_plot.q),
    o3_ppbv,
    o3_gradient,
]

titles = [
    r"$\bar{u}$",
    r"$u$",
    r"$u'$",
    r"$u'_{Kelvin}$",
    r"$T'$",
    r"$T'_{Kelvin}$",
    r"T",
    r"$H_2O$",
    r"$O_3$",
    r"$\frac{\partial O_3}{\partial z}$",
]

cmaps = [
    "seismic",
    "seismic",
    "seismic",
    "seismic",
    cmocean.cm.balance,
    cmocean.cm.balance,
    "coolwarm",
    "YlGnBu",
    cmocean.cm.haline,
    "Spectral_r",
]

norms = [
    Normalize(-20, 20),
    Normalize(-20, 20),
    Normalize(-20, 20),
    Normalize(-20, 20),
    Normalize(-3, 3),
    Normalize(-3, 3),
    Normalize(190, 200),
    Normalize(1.5,5),
    Normalize(80, 300),
    Normalize(0, 300),
]


contours = [
    None,
    S_0,
    None,
    None,
    None,
    None,
    None,
    None,
    None,
    None,
]

contours_clouds = [
    True,
    None,
    None,
    None,
    None,
    None,
    None,
    None,
    None,
    None,
]
pcm_wind = None
pcm_temp = None

Lon= np.broadcast_to(composite_plot.longitude.values, height_plot.shape)


panel_labels = ["a)", "b)", "c)", "d)", "e)", "f)", "g)", "h)", "i)", "j)"]

pcm_list = []

for i, (ax, field, title, shear, cloud, cmap, norm) in enumerate(zip(
    axes.flat, fields, titles, contours, contours_clouds, cmaps, norms
    )):


    pcm = ax.pcolormesh(
        composite_plot.longitude,
        height_plot,
        field,
        cmap=cmap,
        norm=norm,
        shading="auto",
    )

    if shear is not None:
        ax.contour(
            Lon,
            height_plot,
            shear,
            levels=[1],
            colors = ['yellow'],
            linewidths=3,
        )
    if cloud is not None:
        ax.contour(
                Lon,
                height_plot,
                composite_plot.cc,
                cmap = cmocean.cm.rain,
                levels = [0.1,0.2,0.3,0.4,0.5],
                linewidths=3,
        )
    shear_handle = mlines.Line2D(
        [], [], 
        color="yellow",
        linewidth=3,
        label="Ri < 1"
    )

    ax.plot(
        composite_plot.longitude,
        lrt_day,
        "k--",
        lw=2,
        label="LRT"
    )

    # ax.plot(
    #     composite_plot.longitude,
    #     composite_plot.cpt_height,
    #     "k--",
    #     lw=2,
    #     label="LRT"
    # )

    ax.text(
        0.02, 0.97,
        panel_labels[i],
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
        ha="left",
    )

    ax.set_title(title, fontsize=14)
    if i > 7:
        ax.set_xlabel("Relative Longitude (°)", fontsize = 12)
    else:
        ax.set_xticklabels([])
    if i in [0,2,4,6,8]:
        ax.set_ylabel("Height (m)", fontsize = 12)
    ax.grid(alpha=0.3)
    #ax.set_xlim(0,150)
    ax.set_ylim(10000,20000)
    pcm_list.append(pcm)

handles, labels = ax.get_legend_handles_labels()
handles.append(shear_handle)

#axes[0].set_ylabel("Height (m)")
#axes[0].legend(handles = handles)


cbar1 = fig.colorbar(
    pcm_list[0],
    ax=axes[0, :],
    orientation="vertical",
    shrink=0.9,
    pad=0.02,
)
cbar1.set_label(r"Zonal wind [m s$^{-1}$]", fontsize = 11)

cbar2 = fig.colorbar(
    pcm_list[2],
    ax=axes[1, :],
    orientation="vertical",
    shrink=0.9,
    pad=0.02,
)
cbar2.set_label(r"Zonal wind [m s$^{-1}$]", fontsize = 11)

cbar3 = fig.colorbar(
    pcm_list[4],
    ax=axes[2, :],
    orientation="vertical",
    shrink=0.9,
    pad=0.02,
)
cbar3.set_label("Temperature perturbation [K]", fontsize = 11)

# Individual colorbars for the final four panels
cbar_labels = [
    "Temperature [K]",
    "Water vapor [ppmv]",
    "Ozone [ppbv]",
    r"Ozone gradient [ppbv km$^{-1}$]",
]

for j, label in enumerate(cbar_labels):
    ax = axes.flat[j + 6]

    cbar = fig.colorbar(
        pcm_list[j + 6],
        ax=ax,
        orientation="vertical",
        fraction=0.046,
        pad=0.03,
    )
    cbar.set_label(label, fontsize=11)
    cbar.ax.tick_params(labelsize=9)

plt.savefig('../../Plots/PaperReady/AppendixSingleComposite.png', format = 'PNG', dpi = 300)